### Compute the most similar among different modifications

In [ ]:
import sys
sys.path.append('../../utils')
import utility
import math
import itertools
import random
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
from openai.lib._parsing._completions import type_to_response_format_param
import json
import re
from z3 import *
from typing import Literal

load_dotenv()
assert os.getenv("OPENAI_API_KEY") is not None, "You must set your OpenAI API key in the .env file"

openai_client = OpenAI()
seeds_considered = [3, 12, 107, 85, 26]

In [ ]:
folio_dict = utility.make_jsonl_into_list_dictionaries('../../datasets/folio-train-clean.jsonl')
NL_sentences = {}
FOL_formulas = {}
story_id_already_seen = set()
for item in folio_dict:
    story_id = item['story_id']
    if story_id not in story_id_already_seen:
        story_id_already_seen.add(story_id)
        NL_sentences[story_id] = item['premises']
        FOL_formulas[story_id] = item['premises-FOL']
# remove the story 236 because logically problematic (PValue is not used consistently)
NL_sentences.pop(236)
FOL_formulas.pop(236)
# Delete the sentences in the stories that have the Xor symbol ⊕ (it is treated inconsistently in the dataset)
for i in range(2):
    for story_id in FOL_formulas.keys():
        for sentence in FOL_formulas[story_id]:
            if '⊕' in sentence:
                index = FOL_formulas[story_id].index(sentence)
                FOL_formulas[story_id].pop(index)
                NL_sentences[story_id].pop(index)

In [ ]:
# Get the logical modifications for each FOL_formulas
number_logical_modifications = 8
list_modifications_FOLIO = {}
story_exception = []

for story_id in FOL_formulas.keys():
    try: 
        list_modifications_FOLIO[story_id] = []
        for formula in FOL_formulas[story_id]:
            list_formula_mod = []
            # put the ground truth
            list_formula_mod.append(utility.FOLFormula(formula))
            # put the logical modifications
            log_mod = utility.modify_formulas('all_logical', [utility.FOLFormula(formula)], k=number_logical_modifications)
            if log_mod and log_mod[0]:
                log_mod = list(set(log_mod[0]))
                if isinstance(log_mod, utility.FOLFormula):
                    list_formula_mod.append(log_mod)
                else:
                    list_formula_mod.extend(log_mod)
            

            list_modifications_FOLIO[story_id].append(list_formula_mod)

    except:
        story_exception.append(story_id)

print(story_exception)
#utility.save_pickle(list_modifications_FOLIO, '../../perturbations and translations/Most_similar/list_modifications_ms_FOLIO.pkl')

In [ ]:
# Translate the logical modifications
# The code has been designed to be flexible: translation rules can be modified, like in the following 
translation_rules = [
        # Universal quantifier
            {"type": "regex", "pattern": r"∀([a-zA-Z])", "replacements": ["for all \\1,", "for any \\1,"]},
    
        # Existential quantifier
            {"type": "regex", "pattern": r"∃([a-zA-Z])", "replacements": ["there exists \\1 such that", "there is \\1 such that"]},

        # Implication (the premise and the conclusions are between parenthesis)
            {"type": "impl", "pattern": r"\(([^>]+)\)\s*→\s*\(([^>]+)\)", "replacements": ["if \\1, then \\2", "\\1 implies that \\2"]},

        # Equivalence
            {"type": "string", "pattern": "↔", "replacements": ["if and only if ", "is equivalent to "]},

        # Conjunction
            {"type": "string", "pattern": "∧", "replacements": ["and", "moreover"]},
        
        # Disjunction
            {"type": "string", "pattern": "∨", "replacements": ["or"]}]



list_modifications_FOLIO = utility.load_pickle('../../perturbations and translations/Most_similar/list_modifications_ms_FOLIO.pkl')
constant_meaning = utility.load_pickle('../../perturbations and translations/Constant_meaning_FOLIO.pkl')
relational_meaning = utility.load_pickle('../../perturbations and translations/Relational_meaning_FOLIO.pkl')
combination = [0, 1, 0, 1, 0]  # In our experiments, we just consider this combination of translation rules, which corresponds to those specified in the paper's Appendix

list_translated = {}
list_exception = []

for story_id in list_modifications_FOLIO.keys():
    try: 
        list_translated[story_id] = utility.translate_formulas_FOLIO(list_modifications_FOLIO[story_id], constant_meaning[story_id], relational_meaning[story_id]['dict_pos'], relational_meaning[story_id]['dict_neg'], translation_rules, combination)
    except:
        list_exception.append(story_id)

print(list_exception)

# utility.save_pickle(list_translated, 'list_translations_ms_FOLIO.pkl')

In [ ]:
# Shuffle the position and store them for when we will prompt the model
list_new_positions_FOLIO = {}
for story_id in FOL_formulas.keys():
    list_new_positions_FOLIO[story_id] = [[] for i in FOL_formulas[story_id]]
# if list_new_positions_FOLIO[0] = [1, 3, 2], then it means that the sentence list_all_modifications[0][i], in the user prompt is the sentence no. list_new_positions_FOLIO[0][i]. 
    for i in range(len(FOL_formulas[story_id])):
        list_new_positions_FOLIO[story_id][i] = [k+1 for k in range(len(list_modifications_FOLIO[story_id][i]))]
        random.shuffle(list_new_positions_FOLIO[story_id][i])

#utility.save_pickle(list_new_positions_FOLIO, '../../perturbations and translations/Most_similar/list_shuffled_positions_ms_FOLIO.pkl')


In [ ]:
system_prompt_NL = """
You are an expert evaluator specializing in semantic similarity assessment. Your task is to identify the rephrasing that best preserves the original meaning of a given sentence.
Instructions:

- You will receive one original sentence followed by multiple rephrased versions
- Evaluate each rephrasing based solely on semantic/logic equivalence (meaning preservation)
- Ignore differences in grammar, syntax, word order, or writing style
- Select and return the rephrasing that most accurately conveys the same meaning as the original sentence

Evaluation Criteria:

- Prioritize semantic accuracy over grammatical correctness
- Focus on whether the logical meaning is the same or not

Output Format:
Return only the  number of the selected rephrasing that best matches the original sentence's meaning.

Input Format:
Sentence: {sentence}

Rephrasing 1: {rephrasing_1}
Rephrasing 2: {rephrasing_2}
...
Rephrasing n: {rephrasing_n}
"""


system_prompt_FOL = """
You are an expert evaluator specializing in translation into a logical formalism. Your task is to identify the logical formula that best represents the original meaning of a given sentence.
Instructions:

- You will receive one original sentence followed by multiple logical formulas
- Evaluate each formula based solely on semantic/logic equivalence (meaning preservation)
- Select and return the formula that most accurately conveys the same meaning as the original sentence

Evaluation Criteria:
- Focus on whether the logical meaning is the same or not

Output Format:
Return only the number of the selected formula that best matches the original sentence's meaning.

Input Format:
Sentence: {sentence}

Formula 1: {formula_1}
Formula 2: {formula_2}
...
Formula n: {formula_n}
"""

user_prompt_format = """
Sentence: <reference>
<user_prompt_list>
"""

user_prompts_dictionary_NL = {}
user_prompts_dictionary_FOL = {}

# Load the translations
translations = utility.load_pickle('../../perturbations and translations/Most_similar/list_translations_ms_FOLIO.pkl')

list_log_modifications_FOLIO = utility.load_pickle('../../perturbations and translations/Most_similar/list_modifications_ms_FOLIO.pkl')
list_new_positions = utility.load_pickle('../../perturbations and translations/Most_similar/list_shuffled_positions_ms_FOLIO.pkl')


list_exception = []
# Prepare the FOL and the NL prompts
for story_id in FOL_formulas.keys():
    try:
        for i in range(len(FOL_formulas[story_id])):
            # the key (with which we will refer to the question in the future) will be {story_id}_{number_instance}
            key = f'{story_id}_{i}'

            # FOL prompts
            user_prompt = user_prompt_format.replace('<reference>', NL_sentences[story_id][i])
            sentence_list = ''
            for j, item in enumerate(list_new_positions[story_id][i]):
                sentence_list += f'Formula{j+1}: {list_log_modifications_FOLIO[story_id][i][(list_new_positions[story_id][i]).index(j+1)]}\n'
            user_prompt = user_prompt.replace('<user_prompt_list>', sentence_list)
            user_prompts_dictionary_FOL[key] = user_prompt

            # NL prompts
            user_prompt = user_prompt_format.replace('<reference>', NL_sentences[story_id][i])
            sentence_list = ''
            for j, item in enumerate(list_new_positions[story_id][i]):
                transl = translations[story_id][i][(list_new_positions[story_id][i]).index(j+1)]
                sentence_list += f'Rephrasing{j+1}: {transl}\n'
            user_prompt = user_prompt.replace('<user_prompt_list>', sentence_list)
            user_prompts_dictionary_NL[key] = user_prompt
    except:
        list_exception.append(story_id)

print(list_exception)

In [ ]:
#### CALL API: NOT TO RUN
model = 'gpt-4o-mini'
class response_format(BaseModel):
    reasoning: str
    answer: int

all_requests = []

for seed in seeds_considered:
    # for NL
    custom_id_NL = f'FOLIO_most_similar_NL_{seed}'
    requests_NL = utility.create_openai_list_requests_for_batch_completions(model, custom_id_NL, system_prompt_NL, user_prompts_dictionary_NL, type_to_response_format_param(response_format), seed = seed, max_tokens = 2500)

    # for FOL
    custom_id_FOL = f'FOLIO_most_similar_FOL_{seed}'
    requests_FOL = utility.create_openai_list_requests_for_batch_completions(model, custom_id_FOL, system_prompt_FOL, user_prompts_dictionary_FOL, type_to_response_format_param(response_format), seed = seed, max_tokens = 2500)

    all_requests += requests_NL + requests_FOL

# Save the requests in a json file and call openai 
utility.make_list_dictionaries_into_jsonl(all_requests, f'Requests/FOLIO_most_similar_NL_FOL_{model}_seed.jsonl')

# Call API (Remove comments if you want to call API)
# FOLIO_most_similar_NL_FOL_4omini_seed, FOLIO_most_similar_NL_FOL_4omini_batch_seed = utility.call_openai_api_batch_completions(all_requests, 'Requests', f'FOLIO_most_similar_NL_FOL_{model}_seed.jsonl')

# save the batch id
# utility.save_pickle(FOLIO_most_similar_NL_FOL_4omini_seed, f'Requests/batch_ids/FOLIO_most_similar_NL_FOL_{model}_seed.pkl')

In [ ]:
# load the batch
model = 'gpt-4o-mini'
FOLIO_most_similar_NL_FOL_4omini_id_seed = utility.load_pickle(f'Requests/batch_ids/FOLIO_most_similar_NL_FOL_{model}_seed.pkl')
FOLIO_most_similar_NL_FOL_4omini_batch_seed = openai_client.batches.retrieve(FOLIO_most_similar_NL_FOL_4omini_id_seed)
print(FOLIO_most_similar_NL_FOL_4omini_batch_seed.status)
print(FOLIO_most_similar_NL_FOL_4omini_batch_seed)

if utility.extract_batch_errors_openai(FOLIO_most_similar_NL_FOL_4omini_batch_seed):
    print('Error file is present.')
else:
    output = utility.extract_batch_outputs_openai(FOLIO_most_similar_NL_FOL_4omini_batch_seed)
    if not output:
        print('No output file available')

utility.save_pickle(output, f'Results/FOLIO_ms_NL_FOL_{model}_seed.pkl')

In [ ]:
# Clean the results and save them in order to get the clean results and then the performances
model = 'gpt-4o-mini'
output = utility.load_pickle(f'Results/FOLIO_ms_NL_FOL_{model}_seed.pkl')


list_responses = {}
list_error_parsing = []

for seed in seeds_considered:
    list_responses[seed] = []

for item in output:
    number_instance = int(item['custom_id'].split('_')[-1]) 
    story_id = int(item['custom_id'].split('_')[-2])
    seed = int(item['custom_id'].split('_')[-3])
    NLorFOL = str(item['custom_id'].split('_')[-4])
    try: 
        message = json.loads(item['response']['body']['choices'][0]['message']['content'])
        answer = int(message['answer'])
        reasoning = message['reasoning']
        list_responses[seed].append({
                'story_id' : story_id,
                'number_instance' : number_instance,
                'answer': answer,
                'NLorFOL' : NLorFOL,
                'reasoning': reasoning
            })

    except:
        list_error_parsing.append(f'{NLorFOL}_{story_id}_{number_instance}')

print(f'{list_error_parsing=}')
print(len(list_error_parsing))

utility.save_pickle(list_responses, f'Results/Clean Results/FOLIO_ms_NL_FOL_{model}_seed.pkl')


In [ ]:
model = 'gpt-4o-mini'
clean_results = utility.load_pickle( f'Results/Clean Results/FOLIO_ms_NL_FOL_{model}_seed.pkl')
list_new_positions = utility.load_pickle('../../perturbations and translations/Most_similar/list_shuffled_positions_ms_FOLIO.pkl')

most_similar = {}
for seed in seeds_considered:
    most_similar[seed] = {'NL' : {}, 'FOL': {}}
    for story_id in NL_sentences.keys():
        most_similar[seed]['NL'][story_id] = ['' for i in NL_sentences[story_id]]
        most_similar[seed]['FOL'][story_id] = ['' for i in NL_sentences[story_id]]

for seed in seeds_considered:
    for line in clean_results[seed]:
        story_id, i, answer, NLorFOL = line['story_id'], line['number_instance'], line['answer'], line['NLorFOL']
        if answer == list_new_positions[story_id][i][0]:
            most_similar[seed][NLorFOL][story_id][i] = True
        else:
            most_similar[seed][NLorFOL][story_id][i] = False

    total_count = 0
    counter_NL = 0
    counter_NL_false = 0
    for story_id in most_similar[seed]['NL'].keys():
        for item in most_similar[seed]['NL'][story_id]:
            total_count += 1
            if item == True:
                counter_NL += 1
            elif item == False:
                counter_NL_false += 1
    print(model)
    print('Seed:', seed)
    print('NL')
    print(counter_NL, 'right', counter_NL_false, 'wrong, in a total of processed instances', total_count)
    print('Accuracy: ', counter_NL/total_count)

    total_count = 0
    counter_FOL = 0
    counter_FOL_false = 0
    for story_id in most_similar[seed]['FOL'].keys():
        for item in most_similar[seed]['FOL'][story_id]:
            total_count += 1
            if item == True:
                counter_FOL += 1
            elif item == False:
                counter_FOL_false += 1
    print(model)
    print('Seed:', seed)
    print('FOL')
    print(counter_FOL, 'right', counter_FOL_false, 'wrong, in a total of processed instances', total_count)
    print('Accuracy: ', counter_FOL/total_count)


# Do the same with o3-mini

In [ ]:
#### CALL API: NOT TO RUN
model = 'o3-mini'
class response_format(BaseModel):
    reasoning: str
    answer: int

all_requests = []

for seed in seeds_considered:
    # for NL
    custom_id_NL = f'FOLIO_most_similar_NL_{seed}'
    requests_NL = utility.create_openai_list_requests_for_batch_completions(model, custom_id_NL, system_prompt_NL, user_prompts_dictionary_NL, type_to_response_format_param(response_format), seed = seed, max_tokens = 10000)

    # for FOL
    custom_id_FOL = f'FOLIO_most_similar_FOL_{seed}'
    requests_FOL = utility.create_openai_list_requests_for_batch_completions(model, custom_id_FOL, system_prompt_FOL, user_prompts_dictionary_FOL, type_to_response_format_param(response_format), seed = seed, max_tokens = 10000)

    all_requests += requests_NL + requests_FOL

# Save the requests in a json file and call openai 
utility.make_list_dictionaries_into_jsonl(all_requests, f'Requests/FOLIO_most_similar_NL_FOL_{model}_seed.jsonl')

# Call API (Remove comments if you want to call API)
# FOLIO_most_similar_NL_FOL_o3mini_seed, FOLIO_most_similar_NL_FOL_o3mini_batch_seed = utility.call_openai_api_batch_completions(all_requests, 'Requests', f'FOLIO_most_similar_NL_FOL_{model}_seed.jsonl')

# save the batch id
# utility.save_pickle(FOLIO_most_similar_NL_FOL_o3mini_seed, f'Requests/batch_ids/FOLIO_most_similar_NL_FOL_{model}_seed.pkl')

In [ ]:
# load the batch
model = 'o3-mini'
FOLIO_most_similar_NL_FOL_o3mini_id_seed = utility.load_pickle(f'Requests/batch_ids/FOLIO_most_similar_NL_FOL_{model}_seed.pkl')
FOLIO_most_similar_NL_FOL_o3mini_batch_seed = openai_client.batches.retrieve(FOLIO_most_similar_NL_FOL_o3mini_id_seed)
print(FOLIO_most_similar_NL_FOL_o3mini_batch_seed.status)
print(FOLIO_most_similar_NL_FOL_o3mini_batch_seed)

if utility.extract_batch_errors_openai(FOLIO_most_similar_NL_FOL_o3mini_batch_seed):
    print('Error file is present.')
else:
    output = utility.extract_batch_outputs_openai(FOLIO_most_similar_NL_FOL_o3mini_batch_seed)
    if not output:
        print('No output file available')

utility.save_pickle(output, f'Results/FOLIO_ms_NL_FOL_{model}_seed.pkl')

In [ ]:
# Clean the results and save them in order to get the clean results and then the performances
model = 'o3-mini'
output = utility.load_pickle(f'Results/FOLIO_ms_NL_FOL_{model}_seed.pkl')


list_responses = {}
list_error_parsing = []

for seed in seeds_considered:
    list_responses[seed] = []

for item in output:
    number_instance = int(item['custom_id'].split('_')[-1]) 
    story_id = int(item['custom_id'].split('_')[-2])
    seed = int(item['custom_id'].split('_')[-3])
    NLorFOL = str(item['custom_id'].split('_')[-4])
    try: 
        message = json.loads(item['response']['body']['choices'][0]['message']['content'])
        answer = int(message['answer'])
        reasoning = message['reasoning']
        list_responses[seed].append({
                'story_id' : story_id,
                'number_instance' : number_instance,
                'answer': answer,
                'NLorFOL' : NLorFOL,
                'reasoning': reasoning
            })

    except:
        list_error_parsing.append(f'{NLorFOL}_{story_id}_{number_instance}')

print(f'{list_error_parsing=}')
print(len(list_error_parsing))

utility.save_pickle(list_responses, f'Results/Clean Results/FOLIO_ms_NL_FOL_{model}_seed.pkl')


In [ ]:
model = 'o3-mini'
clean_results = utility.load_pickle( f'Results/Clean Results/FOLIO_ms_NL_FOL_{model}_seed.pkl')
list_new_positions = utility.load_pickle('../../perturbations and translations/Most_similar/list_shuffled_positions_ms_FOLIO.pkl')

most_similar = {}
for seed in seeds_considered:
    most_similar[seed] = {'NL' : {}, 'FOL': {}}
    for story_id in NL_sentences.keys():
        most_similar[seed]['NL'][story_id] = ['' for i in NL_sentences[story_id]]
        most_similar[seed]['FOL'][story_id] = ['' for i in NL_sentences[story_id]]

for seed in seeds_considered:
    for line in clean_results[seed]:
        story_id, i, answer, NLorFOL = line['story_id'], line['number_instance'], line['answer'], line['NLorFOL']
        if answer == list_new_positions[story_id][i][0]:
            most_similar[seed][NLorFOL][story_id][i] = True
        else:
            most_similar[seed][NLorFOL][story_id][i] = False

    total_count = 0
    counter_NL = 0
    counter_NL_false = 0
    for story_id in most_similar[seed]['NL'].keys():
        for item in most_similar[seed]['NL'][story_id]:
            total_count += 1
            if item == True:
                counter_NL += 1
            elif item == False:
                counter_NL_false += 1
    print(model)
    print('Seed:', seed)
    print('NL')
    print(counter_NL, 'right', counter_NL_false, 'wrong, in a total of processed instances', total_count)
    print('Accuracy: ', counter_NL/total_count)

    total_count = 0
    counter_FOL = 0
    counter_FOL_false = 0
    for story_id in most_similar[seed]['FOL'].keys():
        for item in most_similar[seed]['FOL'][story_id]:
            total_count += 1
            if item == True:
                counter_FOL += 1
            elif item == False:
                counter_FOL_false += 1
    print(model)
    print('Seed:', seed)
    print('FOL')
    print(counter_FOL, 'right', counter_FOL_false, 'wrong, in a total of processed instances', total_count)
    print('Accuracy: ', counter_FOL/total_count)
